In [1]:

import pandas as pd, openpyxl, os, json, numpy as np
path='CostToGDP.xlsx'
wb=openpyxl.load_workbook(path, data_only=True)
wb.sheetnames

['Cost to GDP']

In [2]:
ws=wb['Cost to GDP']
rows=list(ws.iter_rows(values_only=True))
rows[:20], len(rows), ws.max_row, ws.max_column


([('สัดส่วนต้นทุนโลจิสติกส์ต่อ GDP ของประเทศไทย',
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None),
  ("Thailand's Logistics Cost to GDP",
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None),
  ('หน่วย: ร้อยละ ต่อ GDP',
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   None,
   '   Unit: Percent to GDP'),
  ('ปี',
   2542,
   2543,
   2544,
   2545,
   2546,
   2547,
   2548,
   2549,
   2550,
   2551,
   2552,
   2553,
   2554,
   2

In [3]:
import re

rows=list(ws.iter_rows(values_only=True))
header=rows[3][1:27]
norm_years=[]
for y in header:
    if y is None: 
        norm_years.append(None)
    elif isinstance(y,(int,float)):
        norm_years.append(int(y))
    else:
        m=re.search(r'(\d{4})', str(y))
        norm_years.append(int(m.group(1)) if m else None)
norm_years[:]

[2542,
 2543,
 2544,
 2545,
 2546,
 2547,
 2548,
 2549,
 2550,
 2551,
 2552,
 2553,
 2554,
 2555,
 2556,
 2557,
 2558,
 2559,
 2560,
 2561,
 2562,
 2563,
 2564,
 2565,
 2566,
 2567]

In [4]:
labels={}
for r in rows[5:]:
    label=r[0]
    if label is None:
        continue
    year_map={}
    vals=r[1:27]
    for y,val in zip(norm_years, vals):
        year_map[y]=val
    labels[label]=year_map

years_buddhist=[2552,2554,2556,2558,2560,2562,2564,2566]
overall=[labels['สัดส่วนต้นทุนโลจิสติกส์ ต่อ GDP'][y] for y in years_buddhist]
transport=[labels['สัดส่วนต้นทุนการขนส่งสินค้า ต่อ GDP'][y] for y in years_buddhist]
inventory=[labels['สัดส่วนต้นทุนการเก็บรักษาสินค้าคงคลัง ต่อ GDP'][y] for y in years_buddhist]
admin=[labels['สัดส่วนต้นทุนการบริหารจัดการ ต่อ GDP'][y] for y in years_buddhist]
overall, transport, inventory, admin


([15.1,
  14.719557424617664,
  14.233262229794475,
  13.889495708687388,
  13.5,
  13.347034336124427,
  14.285123153139681,
  14.15775641090331],
 [7.3,
  7.5198873574911795,
  7.500000000000001,
  7.3999999999999995,
  6.7,
  6.6,
  7.037050337642299,
  6.743612025427742],
 [6.4,
  5.861528483070333,
  5.526231526915224,
  5.2,
  5.7,
  5.759463918183568,
  6.1850184823685135,
  6.36056828825544],
 [1.4,
  1.3381415840561512,
  1.3070307028792512,
  1.2894957086873873,
  1,
  0.9932446877308441,
  1.063054333128869,
  1.053576097220128])

In [5]:
years=years_buddhist
data = {
'รวมรายได้ทั้งสิ้นต่อเดือน_amt':[20903,23223,25194,26915,26946,26018,27352,29030],
'รวมรายได้ที่เป็นตัวเงิน_amt':[17512,19699,21562,22913,22877,22050,22367,24501],
'ค่าจ้างและเงินเดือน_amt':[8418,8942,10276,11967,12150,11686,11639,12605],
'กำไรสุทธิจากการทำธุรกิจ_amt':[4246,4666,4648,5096,4882,4704,4319,5417],
'กำไรสุทธิจากการทำเกษตร_amt':[2390,3112,3665,2587,2327,2103,2297,2443],
'บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_amt':[483,637,570,723,998,957,1210,1275],
'เงินชดเชยการออกจากงาน_amt':[14,21,11,6,23,18,48,12],
'เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ_amt':[1634,1868,2064,2129,2111,2355,2610,2507],
'รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร_amt':[188,254,180,222,211,116,121,141],
'ดอกเบี้ยและเงินปันผล_amt':[139,199,148,183,175,111,123,101],

'รวมรายได้ทั้งสิ้นต่อเดือน_pct':[100.00,100.0,100.0,100.0,100.0,100.0,100.0,100.0],
'รวมรายได้ที่เป็นตัวเงิน_pct':[83.78,84.82,85.58,85.13,84.90,84.75,81.77,84.40],
'ค่าจ้างและเงินเดือน_pct':[40.27,38.50,40.79,44.46,45.09,44.92,42.55,43.42],
'กำไรสุทธิจากการทำธุรกิจ_pct':[20.31,20.09,18.45,18.93,18.12,18.08,15.79,18.66],
'กำไรสุทธิจากการทำเกษตร_pct':[11.43,13.40,14.55,9.61,8.64,8.08,8.40,8.42],
'บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_pct':[2.31,2.74,2.26,2.69,3.70,3.68,4.42,4.39],
'เงินชดเชยการออกจากงาน_pct':[0.07,0.09,0.04,0.02,0.09,0.07,0.18,0.04],
'เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ_pct':[7.82,8.04,8.19,7.91,7.83,9.05,9.54,8.64],
'รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร_pct':[0.90,1.09,0.71,0.82,0.78,0.45,0.44,0.49],
'ดอกเบี้ยและเงินปันผล_pct':[0.66,0.86,0.59,0.68,0.65,0.43,0.45,0.35],
}
df=pd.DataFrame(data,index=years)
df['CostToGDP']=overall
df['TransportCostToGDP']=transport
df['InventoryCostToGDP']=inventory
df['AdminCostToGDP']=admin
df.head()


,รวมรายได้ทั้งสิ้นต่อเดือน_amt,รวมรายได้ที่เป็นตัวเงิน_amt,ค่าจ้างและเงินเดือน_amt,กำไรสุทธิจากการทำธุรกิจ_amt,กำไรสุทธิจากการทำเกษตร_amt,บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_amt,เงินชดเชยการออกจากงาน_amt,เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ_amt,รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร_amt,ดอกเบี้ยและเงินปันผล_amt,...,กำไรสุทธิจากการทำเกษตร_pct,บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_pct,เงินชดเชยการออกจากงาน_pct,เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ_pct,รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร_pct,ดอกเบี้ยและเงินปันผล_pct,CostToGDP,TransportCostToGDP,InventoryCostToGDP,AdminCostToGDP
2552,20903,17512,8418,4246,2390,483,14,1634,188,139,...,11.43,2.31,0.07,7.82,0.90,0.66,15.100000,7.300000,6.400000,1.400000
2554,23223,19699,8942,4666,3112,637,21,1868,254,199,...,13.40,2.74,0.09,8.04,1.09,0.86,14.719557,7.519887,5.861528,1.338142
2556,25194,21562,10276,4648,3665,570,11,2064,180,148,...,14.55,2.26,0.04,8.19,0.71,0.59,14.233262,7.500000,5.526232,1.307031
2558,26915,22913,11967,5096,2587,723,6,2129,222,183,...,9.61,2.69,0.02,7.91,0.82,0.68,13.889496,7.400000,5.200000,1.289496
2560,26946,22877,12150,4882,2327,998,23,2111,211,175,...,8.64,3.70,0.09,7.83,0.78,0.65,13.500000,6.700000,5.700000,1.000000


## ตรวจดูความเกี่ยวข้อง (correlation) ของข้อมูล

### วิธีแปลผล

ค่า pearson_r และ spearman_rho เอาไว้ใช้ดูว่าค่าเหล่านี้ correlate กันหรือไม่

* ถ้าค่าเป็นบวก แปลว่า x เพิ่ม y เพิ่ม
* ถ้าค่าเป็นลบ แปลว่า x ลบ y ลบ
* จะดูว่าค่า correlate กันมากมั้ย ให้ดูว่าค่าใกล้ 1 หรือ -1 มั้ย

ค่า p-value เอาไว้เช็คว่าค่าที่ได้มามัน **เป็นไปได้จริงๆใช่มั้ย**

* p < 0.05 = มีโอกาสเป็นไปได้สูงว่าเป็นจริง
* p > 0.5 = อาจจะมีความเป็นไปได้ว่า ค่า correlate สูง อาจจะเกิดขึ้นเพราะความบังเอิญ

In [6]:
from scipy.stats import pearsonr, spearmanr
results=[]
for col in data:
    x=df[col].astype(float).values
    y=df['CostToGDP'].astype(float).values
    r,p=pearsonr(x,y)
    rs,ps=spearmanr(x,y)
    results.append([col,r,p,rs,ps])
res=pd.DataFrame(results,columns=['variable','pearson_r','pearson_p','spearman_rho','spearman_p'])
res.sort_values('pearson_r')


C:\Users\USER\AppData\Local\Temp\ipykernel_10224\2596602981.py:6: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r,p=pearsonr(x,y)
C:\Users\USER\AppData\Local\Temp\ipykernel_10224\2596602981.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rs,ps=spearmanr(x,y)


,variable,pearson_r,pearson_p,spearman_rho,spearman_p
12,ค่าจ้างและเงินเดือน_pct,-0.873994,0.004541,-0.928571,0.000863
2,ค่าจ้างและเงินเดือน_amt,-0.803182,0.016358,-0.761905,0.028005
1,รวมรายได้ที่เป็นตัวเงิน_amt,-0.720630,0.043727,-0.619048,0.101733
0,รวมรายได้ทั้งสิ้นต่อเดือน_amt,-0.679042,0.064038,-0.476190,0.232936
7,เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ_amt,-0.548794,0.158947,-0.476190,0.232936
5,บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_amt,-0.486312,0.221730,-0.476190,0.232936
3,กำไรสุทธิจากการทำธุรกิจ_amt,-0.461179,0.250075,-0.690476,0.057990
15,บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_pct,-0.439708,0.275652,-0.261905,0.530923
11,รวมรายได้ที่เป็นตัวเงิน_pct,-0.290410,0.485322,-0.380952,0.351813
17,เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ_pct,-0.245597,0.557688,-0.214286,0.610344


In [7]:
res['abs_r']=res['pearson_r'].abs()
res2=res[~res['pearson_r'].isna()].sort_values('abs_r', ascending=False)
res2.round(4)


,variable,pearson_r,pearson_p,spearman_rho,spearman_p,abs_r
12,ค่าจ้างและเงินเดือน_pct,-0.8740,0.0045,-0.9286,0.0009,0.8740
2,ค่าจ้างและเงินเดือน_amt,-0.8032,0.0164,-0.7619,0.0280,0.8032
1,รวมรายได้ที่เป็นตัวเงิน_amt,-0.7206,0.0437,-0.6190,0.1017,0.7206
0,รวมรายได้ทั้งสิ้นต่อเดือน_amt,-0.6790,0.0640,-0.4762,0.2329,0.6790
14,กำไรสุทธิจากการทำเกษตร_pct,0.5801,0.1317,0.5476,0.1600,0.5801
7,เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ_amt,-0.5488,0.1589,-0.4762,0.2329,0.5488
18,รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่น...,0.5137,0.1929,0.4286,0.2894,0.5137
5,บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_amt,-0.4863,0.2217,-0.4762,0.2329,0.4863
13,กำไรสุทธิจากการทำธุรกิจ_pct,0.4775,0.2315,0.5476,0.1600,0.4775
3,กำไรสุทธิจากการทำธุรกิจ_amt,-0.4612,0.2501,-0.6905,0.0580,0.4612


In [8]:
res_nonconst=res[~res['pearson_r'].isna()].copy()
res_nonconst['direction']=np.where(res_nonconst['pearson_r']>=0,'positive','negative')
top_pos=res_nonconst.sort_values('pearson_r', ascending=False).head(10)[['variable','pearson_r','pearson_p','spearman_rho','spearman_p']]
top_neg=res_nonconst.sort_values('pearson_r', ascending=True).head(10)[['variable','pearson_r','pearson_p','spearman_rho','spearman_p']]
top_pos.round(4), top_neg.round(4)


(                                             variable  pearson_r  pearson_p  \
 14                         กำไรสุทธิจากการทำเกษตร_pct     0.5801     0.1317   
 18  รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่น...     0.5137     0.1929   
 13                        กำไรสุทธิจากการทำธุรกิจ_pct     0.4775     0.2315   
 19                           ดอกเบี้ยและเงินปันผล_pct     0.3925     0.3361   
 4                          กำไรสุทธิจากการทำเกษตร_amt     0.3392     0.4110   
 8   รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่น...     0.3036     0.4648   
 9                            ดอกเบี้ยและเงินปันผล_amt     0.1365     0.7473   
 16                          เงินชดเชยการออกจากงาน_pct     0.1161     0.7843   
 6                           เงินชดเชยการออกจากงาน_amt     0.0222     0.9584   
 17        เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ_pct    -0.2456     0.5577   
 
     spearman_rho  spearman_p  
 14        0.5476      0.1600  
 18        0.4286      0.2894  
 13        0.5476     

In [ ]:
def corr_with(target):
    out=[]
    for col in data:
        x=df[col].astype(float).values
        y=df[target].astype(float).values
        try:
            r,p=pearsonr(x,y)
        except Exception:
            r,p=np.nan,np.nan
        try:
            rs,ps=spearmanr(x,y)
        except Exception:
            rs,ps=np.nan,np.nan
        out.append([col,r,p,rs,ps])
    return pd.DataFrame(out,columns=['variable','pearson_r','pearson_p','spearman_rho','spearman_p'])

for target in ['TransportCostToGDP','InventoryCostToGDP','AdminCostToGDP']:
    tmp=corr_with(target)
    print(target)
    print(tmp[['variable','pearson_r']].sort_values('pearson_r').head(5).round(4).to_string(index=False))
    print(tmp[['variable','pearson_r']].sort_values('pearson_r', ascending=False).head(5).round(4).to_string(index=False))
    print()


TransportCostToGDP
                                     variable  pearson_r
บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_pct    -0.7830
บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_amt    -0.7603
                      ค่าจ้างและเงินเดือน_pct    -0.7453
                      ค่าจ้างและเงินเดือน_amt    -0.6730
  เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ_amt    -0.5872
                                                                    variable  pearson_r
                                                  กำไรสุทธิจากการทำเกษตร_pct     0.8364
                                                  กำไรสุทธิจากการทำเกษตร_amt     0.7475
                                                    ดอกเบี้ยและเงินปันผล_pct     0.6929
รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร_pct     0.6860
รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร_amt     0.6238

InventoryCostToGDP
                                                                    variable  pearson_r
   

C:\Users\USER\AppData\Local\Temp\ipykernel_10224\662798649.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r,p=pearsonr(x,y)
C:\Users\USER\AppData\Local\Temp\ipykernel_10224\662798649.py:11: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rs,ps=spearmanr(x,y)
C:\Users\USER\AppData\Local\Temp\ipykernel_10224\662798649.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r,p=pearsonr(x,y)
C:\Users\USER\AppData\Local\Temp\ipykernel_10224\662798649.py:11: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rs,ps=spearmanr(x,y)
C:\Users\USER\AppData\Local\Temp\ipykernel_10224\662798649.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r,p=pearsonr(x,y)
C:\Users\USER\AppData\Local\Temp\ipykernel_10224\662798649.py:11: ConstantInputWarning: An input 

In [13]:
from IPython.display import display, HTML

def display_corr_analysis(target):
    # 1. Get the data
    tmp = corr_with(target)
    
    # 2. Create Top 5 and Bottom 5 DataFrames
    bottom_5 = tmp[['variable', 'pearson_r']].sort_values('pearson_r').head(5)
    top_5 = tmp[['variable', 'pearson_r']].sort_values('pearson_r', ascending=False).head(5)
    
    # 3. Apply Styling (Heatmap color)
    def style_df(df, title):
        return df.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1)\
                       .set_caption(f"<b>{title}</b>")\
                       .format({'pearson_r': "{:.4f}"})\
                       .hide(axis='index')

    # 4. Display Header and Tables
    display(HTML(f"### Analysis for: <span style='color: #2e86de;'>{target}</span>"))
    
    # This displays the two tables side-by-side
    t1 = style_df(top_5, "Strongest Positive Correlation")._repr_html_()
    t2 = style_df(bottom_5, "Strongest Negative Correlation")._repr_html_()
    
    display(HTML(f"""
    <div style="display: flex; gap: 50px;">
        <div>{t1}</div>
        <div>{t2}</div>
    </div>
    <hr>
    """))

# Run the improved display
for target in ['TransportCostToGDP', 'InventoryCostToGDP', 'AdminCostToGDP']:
    display_corr_analysis(target)

C:\Users\USER\AppData\Local\Temp\ipykernel_10224\662798649.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r,p=pearsonr(x,y)
C:\Users\USER\AppData\Local\Temp\ipykernel_10224\662798649.py:11: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rs,ps=spearmanr(x,y)


variable,pearson_r
กำไรสุทธิจากการทำเกษตร_pct,0.8364
กำไรสุทธิจากการทำเกษตร_amt,0.7475
ดอกเบี้ยและเงินปันผล_pct,0.6929
รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร_pct,0.6860
รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร_amt,0.6238
variable,pearson_r
บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_pct,-0.7830
บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_amt,-0.7603
ค่าจ้างและเงินเดือน_pct,-0.7453
ค่าจ้างและเงินเดือน_amt,-0.6730


C:\Users\USER\AppData\Local\Temp\ipykernel_10224\662798649.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r,p=pearsonr(x,y)
C:\Users\USER\AppData\Local\Temp\ipykernel_10224\662798649.py:11: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rs,ps=spearmanr(x,y)


variable,pearson_r
เงินชดเชยการออกจากงาน_pct,0.4152
บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_pct,0.3945
เงินชดเชยการออกจากงาน_amt,0.3673
เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ_pct,0.3199
บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_amt,0.3137
variable,pearson_r
รวมรายได้ที่เป็นตัวเงิน_pct,-0.6286
ดอกเบี้ยและเงินปันผล_amt,-0.5764
รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร_amt,-0.4136
ดอกเบี้ยและเงินปันผล_pct,-0.3698


C:\Users\USER\AppData\Local\Temp\ipykernel_10224\662798649.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r,p=pearsonr(x,y)
C:\Users\USER\AppData\Local\Temp\ipykernel_10224\662798649.py:11: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rs,ps=spearmanr(x,y)


variable,pearson_r
กำไรสุทธิจากการทำเกษตร_pct,0.8019
รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร_pct,0.7557
กำไรสุทธิจากการทำธุรกิจ_pct,0.6885
ดอกเบี้ยและเงินปันผล_pct,0.6853
รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร_amt,0.6179
variable,pearson_r
บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_pct,-0.8785
บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์_amt,-0.8750
ค่าจ้างและเงินเดือน_amt,-0.8094
เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ_amt,-0.7894


In [10]:
# Build nice tables
res_nonconst = res[~res['pearson_r'].isna()].copy()
# separate amount/pct
amount_res = res_nonconst[res_nonconst['variable'].str.endswith('_amt')].copy()
pct_res = res_nonconst[res_nonconst['variable'].str.endswith('_pct')].copy()
for tbl in [amount_res,pct_res]:
    tbl['รายการ']=tbl['variable'].str.replace(r'_(amt|pct)$','', regex=True)
amount_top = amount_res[['รายการ','pearson_r','pearson_p','spearman_rho','spearman_p']].sort_values('pearson_r')
pct_top = pct_res[['รายการ','pearson_r','pearson_p','spearman_rho','spearman_p']].sort_values('pearson_r')
amount_top.round(3), pct_top.round(3)


(                                              รายการ  pearson_r  pearson_p  \
 2                                ค่าจ้างและเงินเดือน     -0.803      0.016   
 1                            รวมรายได้ที่เป็นตัวเงิน     -0.721      0.044   
 0                          รวมรายได้ทั้งสิ้นต่อเดือน     -0.679      0.064   
 7            เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ     -0.549      0.159   
 5          บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์     -0.486      0.222   
 3                            กำไรสุทธิจากการทำธุรกิจ     -0.461      0.250   
 6                              เงินชดเชยการออกจากงาน      0.022      0.958   
 9                               ดอกเบี้ยและเงินปันผล      0.136      0.747   
 8  รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่น...      0.304      0.465   
 4                             กำไรสุทธิจากการทำเกษตร      0.339      0.411   
 
    spearman_rho  spearman_p  
 2        -0.762       0.028  
 1        -0.619       0.102  
 0        -0.476       0.233  
 7   

In [11]:
def fmt(df):
    return df.assign(
        pearson_r=df['pearson_r'].round(3),
        pearson_p=df['pearson_p'].round(3),
        spearman_rho=df['spearman_rho'].round(3),
        spearman_p=df['spearman_p'].round(3),
    )
print("Amounts")
print(fmt(amount_top)[['รายการ','pearson_r','pearson_p','spearman_rho','spearman_p']].to_string(index=False))
print("\nPercents")
print(fmt(pct_top)[['รายการ','pearson_r','pearson_p','spearman_rho','spearman_p']].to_string(index=False))


Amounts
                                                                  รายการ  pearson_r  pearson_p  spearman_rho  spearman_p
                                                     ค่าจ้างและเงินเดือน     -0.803      0.016        -0.762       0.028
                                                 รวมรายได้ที่เป็นตัวเงิน     -0.721      0.044        -0.619       0.102
                                               รวมรายได้ทั้งสิ้นต่อเดือน     -0.679      0.064        -0.476       0.233
                                 เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ     -0.549      0.159        -0.476       0.233
                               บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์     -0.486      0.222        -0.476       0.233
                                                 กำไรสุทธิจากการทำธุรกิจ     -0.461      0.250        -0.690       0.058
                                                   เงินชดเชยการออกจากงาน      0.022      0.958         0.095       0.823
                        

In [12]:
import numpy as np
cost_series=pd.Series(overall,index=years)
cost_series.round(3), cost_series.iloc[-1]-cost_series.iloc[0], (cost_series.iloc[-1]/cost_series.iloc[0]-1)*100


(2552    15.100
 2554    14.720
 2556    14.233
 2558    13.889
 2560    13.500
 2562    13.347
 2564    14.285
 2566    14.158
 dtype: float64,
 np.float64(-0.9422435890966891),
 np.float64(-6.240023768852243))

In [17]:
from IPython.display import display, HTML

def style_correlation_table(df, title):
    """
    Applies professional formatting to correlation dataframes.
    Updated for Pandas 2.1+ (uses .map instead of .applymap)
    """
    # 1. Define the styling function for p-values
    def highlight_significant(val):
        try:
            color = 'green' if val < 0.05 else 'black'
            weight = 'bold' if val < 0.05 else 'normal'
            return f'color: {color}; font-weight: {weight}'
        except:
            return ''

    # 2. Apply the style
    # Note: .map replaces .applymap in modern Pandas
    styled = (df.style
        .background_gradient(cmap='coolwarm', subset=['pearson_r', 'spearman_rho'], vmin=-1, vmax=1)
        .map(highlight_significant, subset=['pearson_p', 'spearman_p'])
        .format({
            'pearson_r': '{:.3f}',
            'pearson_p': '{:.3f}',
            'spearman_rho': '{:.3f}',
            'spearman_p': '{:.3f}'
        })
        .set_caption(f"<h3 style='text-align:left; color:#2c3e50;'>{title}</h3>")
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#f8f9fa'), ('color', '#333'), ('font-weight', 'bold')]},
            {'selector': 'tr:hover', 'props': [('background-color', '#ffff99')]} 
        ])
        .hide(axis='index')
    )
    return styled

# --- Execute and Display ---
display(HTML("<h2>Correlation Analysis: Household Income vs. Logistics Cost</h2>"))

# Display Amounts
amount_styled = style_correlation_table(amount_top, "Correlation by Absolute Amounts (THB)")
display(amount_styled)

display(HTML("<br><hr><br>"))

# Display Percentages
pct_styled = style_correlation_table(pct_top, "Correlation by Income Proportions (%)")
display(pct_styled)

รายการ,pearson_r,pearson_p,spearman_rho,spearman_p
ค่าจ้างและเงินเดือน,-0.803,0.016,-0.762,0.028
รวมรายได้ที่เป็นตัวเงิน,-0.721,0.044,-0.619,0.102
รวมรายได้ทั้งสิ้นต่อเดือน,-0.679,0.064,-0.476,0.233
เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ,-0.549,0.159,-0.476,0.233
บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์,-0.486,0.222,-0.476,0.233
กำไรสุทธิจากการทำธุรกิจ,-0.461,0.250,-0.690,0.058
เงินชดเชยการออกจากงาน,0.022,0.958,0.095,0.823
ดอกเบี้ยและเงินปันผล,0.136,0.747,0.190,0.651
รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร,0.304,0.465,0.286,0.493
กำไรสุทธิจากการทำเกษตร,0.339,0.411,0.381,0.352


รายการ,pearson_r,pearson_p,spearman_rho,spearman_p
ค่าจ้างและเงินเดือน,-0.874,0.005,-0.929,0.001
บำเหน็จ บำนาญ เบี้ยหวัด หรือเงินสงเคราะห์,-0.440,0.276,-0.262,0.531
รวมรายได้ที่เป็นตัวเงิน,-0.290,0.485,-0.381,0.352
เงินช่วยเหลือจากบุคคลนอกครัวเรือนและรัฐ,-0.246,0.558,-0.214,0.610
เงินชดเชยการออกจากงาน,0.116,0.784,0.255,0.543
ดอกเบี้ยและเงินปันผล,0.393,0.336,0.405,0.320
กำไรสุทธิจากการทำธุรกิจ,0.477,0.231,0.548,0.160
รายรับจากการให้เช่าห้อง/ที่ดินและสินทรัพย์อื่นๆ ค่าลิขสิทธิ์และสิทธิบัตร,0.514,0.193,0.429,0.289
กำไรสุทธิจากการทำเกษตร,0.580,0.132,0.548,0.160


# สรุปผลการหาความสัมพันธ์:

### จาก ตารางที่ 1.1 : จำนวนรายได้เฉลี่ยต่อเดือนของครัวเรือน และร้อยละ จำแนกตามแหล่งที่มาของรายได้ เทียบกับ CostToGDP เราสามารถสรุปได้ว่า

1. เมื่อรายได้ครัวเรือนมาจากฐานเงินเดือน (Wages) มากขึ้น สัดส่วน CostToGDP มักจะต่ำลง
2. สัดส่วนของรายได้เกษตรอาจจะมีส่วนทำให้ต้นทุน GDP สูงขึ้น